# Classification Model Comparison with mlcore

This notebook compares multiple classification algorithms using the same dataset and evaluation workflow.

In [1]:
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import mlcore.classification

from mlcore.core.registry import MODEL_REGISTRY
from mlcore.core.trainer import Trainer
from mlcore.evaluation.classification import evaluate_classification

In [2]:
X, y = make_classification(
    n_samples=2000,
    n_features=60,
    n_informative=25,
    n_redundant=10,
    n_classes=2,
    class_sep=1.2,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

In [3]:
trainer = Trainer(
    registry=MODEL_REGISTRY,
)

In [4]:
models = {
    "Logistic Regression": {
        "algorithm": "logistic_regression",
        "params": {
            "max_iter": 1000,
        },
    },
    "Random Forest": {
        "algorithm": "random_forest",
        "params": {
            "n_estimators": 200,
            "random_state": 42,
        },
    },
    "Extra Trees": {
        "algorithm": "extra_trees",
        "params": {
            "n_estimators": 200,
            "random_state": 42,
        },
    },
    "XGBoost": {
        "algorithm": "xgb_classifier",
        "params": {
            "n_estimators": 200,
            "max_depth": 5,
            "random_state": 42,
            "verbosity": 0,
        },
    },
    "LightGBM": {
        "algorithm": "lgbm_classifier",
        "params": {
            "n_estimators": 200,
            "random_state": 42,
            "verbose": -1,
        },
    },
}

In [5]:
results = []

for model_name, config in models.items():
    result = trainer.fit(
        algorithm=config["algorithm"],
        X=X_train,
        y=y_train,
        **config["params"],
    )

    predictions = trainer.predict(
        result,
        X_test,
    )

    metrics = evaluate_classification(
        y_true=y_test,
        y_pred=predictions,
    )

    metrics["model"] = model_name
    metrics["algorithm"] = config["algorithm"]
    metrics["backend"] = result.spec.backend

    results.append(metrics)

/home/david/miniconda3/envs/ml_library_dev/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/david/miniconda3/envs/ml_library_dev/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/david/miniconda3/envs/ml_library_dev/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
comparison = pd.DataFrame(results)

comparison = comparison[
    [
        "model",
        "backend",
        "algorithm",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "mcc",
    ]
]

comparison

,model,backend,algorithm,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc
0,Logistic Regression,sklearn,logistic_regression,0.795,0.795,0.801020,0.785,0.805,0.792929,0.590118
1,Random Forest,sklearn,random_forest,0.920,0.920,0.924242,0.915,0.925,0.919598,0.840042
2,Extra Trees,sklearn,extra_trees,0.925,0.925,0.920792,0.930,0.920,0.925373,0.850043
3,XGBoost,xgboost,xgb_classifier,0.935,0.935,0.930693,0.940,0.930,0.935323,0.870044
4,LightGBM,lightgbm,lgbm_classifier,0.935,0.935,0.939394,0.930,0.940,0.934673,0.870044


In [7]:
comparison.sort_values(
    by="f1",
    ascending=False,
)

,model,backend,algorithm,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc
3,XGBoost,xgboost,xgb_classifier,0.935,0.935,0.930693,0.940,0.930,0.935323,0.870044
4,LightGBM,lightgbm,lgbm_classifier,0.935,0.935,0.939394,0.930,0.940,0.934673,0.870044
2,Extra Trees,sklearn,extra_trees,0.925,0.925,0.920792,0.930,0.920,0.925373,0.850043
1,Random Forest,sklearn,random_forest,0.920,0.920,0.924242,0.915,0.925,0.919598,0.840042
0,Logistic Regression,sklearn,logistic_regression,0.795,0.795,0.801020,0.785,0.805,0.792929,0.590118


In [8]:
[
    spec.name
    for spec in MODEL_REGISTRY.filter(
        task="classification",
        tags=["boosting"],
    )
]

['lgbm_classifier', 'xgb_classifier', 'xgb_rf_classifier']

In [9]:
best_model_name = comparison.sort_values(
    by="f1",
    ascending=False,
).iloc[0]["model"]

best_model_name

'XGBoost'

## Summary

In this notebook we:

- Trained several classification algorithms through the same `Trainer` interface.
- Compared scikit-learn, XGBoost, and LightGBM models.
- Evaluated all models with the same classification metric function.
- Used registry metadata to track backend and algorithm identity.

This notebook shows how `mlcore` can be used as a unified classical machine learning benchmarking layer.